# AI Timetable Scheduler — Class Presentation Guide (Hinglish)

Yeh notebook **Google Colab** ke liye taiyaar hai. Ismein project ke actual dataset loaders, constraint checker aur chhe search algorithms use hote hain. Charts aap cells run karoge tab generate honge.

> **Colab mein pehle:** notebook ka pehla setup code cell run karo. Required project source code notebook ke andar embedded hai—alag ZIP upload nahi karni.
> **Dataset:** `DATASET_NAME` cell mein `"small"` (default), `"large"`, ya `"iiit"` select karo. Bade dataset par search algorithms ko zyada waqt lag sakta hai.

Notebook pehle `matplotlib` install karegi, phir embedded `core`, `data`, aur `solvers` source files ko Colab runtime mein extract karegi.

In [ ]:
# Google Colab setup: project source notebook ke andar embedded hai. ZIP upload ki zaroorat nahi.
!pip -q install matplotlib

import base64
from io import BytesIO
from pathlib import Path
import sys
import zipfile

PROJECT_ROOT = Path("/content/timetable_project")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE_B64 = (
    "..."
    "AwAAynAAAAAA"
)
source_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
with zipfile.ZipFile(BytesIO(source_bytes)) as source_zip:
    source_zip.extractall(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project source embedded notebook se extract ho gaya:", PROJECT_ROOT)

In [ ]:
# Setup aur actual project modules import
# Agar matplotlib missing ho to terminal mein: pip install matplotlib

import time
import matplotlib.pyplot as plt

from data.datasets import (
    get_small_dataset,
    get_large_dataset,
    get_iiit_kota_odd_sem_dataset,
)
from core.constraints import ConstraintChecker
from solvers.backtracking import PlainBacktrackingSolver, HeuristicBacktrackingSolver
from solvers.min_conflicts import MinConflictsSolver
from solvers.simulated_annealing import SimulatedAnnealingSolver
from solvers.hybrid import HybridCSPSolver
from solvers.hill_climbing import HillClimbingSolver

DATASET_NAME = "small"  # options: "small", "large", "iiit"

loaders = {
    "small": get_small_dataset,
    "large": get_large_dataset,
    "iiit": get_iiit_kota_odd_sem_dataset,
}
if DATASET_NAME not in loaders:
    raise ValueError('DATASET_NAME "small", "large", ya "iiit" hona chahiye.')

problem = loaders[DATASET_NAME]()
checker = ConstraintChecker(problem)
print("Loaded dataset:", problem.name)

## 1. Hum kaunsa data use kar rahe hain?

App mein teen built-in preset datasets hain: **Small**, **Large**, aur **IIIT Kota Odd Semesters**. Custom JSON upload ko `TimetableProblem.from_dict()` Python objects mein badalta hai aur uske baad session variables banata hai. Is notebook mein wahi built-in loader functions directly use ho rahe hain.

Har dataset mein yeh information hoti hai:

- **Classes/batches:** class ka naam, student strength, required subjects aur lunch period.
- **Subjects:** subject code/name, weekly hours, theory ya lab, aur consecutive lab requirement.
- **Teachers:** kaunse subjects padha sakte hain, maximum weekly load, unavailable aur preferred slots.
- **Rooms:** room capacity aur classroom/lab type.
- **Time structure:** weekdays, periods/day aur lunch/break periods.

Weekly subject requirement ko alag-alag **session variables** mein todte hain. Isliye session variables ki total ginti sirf subject count nahi hoti; har class ke weekly hours add hote hain.

In [ ]:
# Dataset ka size aur har class ke sessions print karein
print(f"Dataset: {problem.name}")
print(f"Classes: {len(problem.classes)}")
print(f"Teachers: {len(problem.teachers)}")
print(f"Subjects: {len(problem.subjects)}")
print(f"Rooms: {len(problem.rooms)}")
print(f"Days: {len(problem.time_structure.days)}")
print(f"Periods per day: {problem.time_structure.periods_per_day}")
print(f"Generated session variables: {len(problem.variables)}")

print("\nHar class ke sessions:")
for class_name in problem.classes:
    count = sum(v.class_name == class_name for v in problem.variables)
    print(f"  {class_name}: {count}")

# Project data ke expected reference counts
reference_counts = {
    "Small_Dataset_3x6x4": "3 classes, 6 teachers, 6 subjects, 4 rooms; 5 x 6 timetable; 47 sessions",
    "Large_Dataset_8x7x9": "8 classes, 16 teachers, 9 subjects, 9 rooms; 5 x 7 timetable; 132 sessions",
    "IIIT_Kota_Odd_Semesters_2026": "4 batches, 13 teachers, 35 subjects, 13 rooms; 5 x 8 timetable; 97 sessions",
}
print("\nDataset reference:", reference_counts.get(problem.name, "Custom dataset"))
print("Lunch/break periods:", problem.time_structure.break_periods)

is_valid, validation_errors = problem.validate_input()
print("\nBasic input validation:", "PASS" if is_valid else "CHECK REQUIRED")
for error in validation_errors:
    print(" -", error)

## 2. Ek session variable ka domain kaise banta hai?

Har session variable ko `(day, period, room)` ke possible combinations milte hain. `get_domain_for_variable()` invalid combinations pehle hi filter karta hai:

1. Room class ki strength ke liye kaafi bada ho.
2. Theory session classroom mein aur lab session lab mein ho.
3. Class ke lunch period mein class na ho.
4. Teacher ke unavailable waqt par session na ho.
5. Multi-hour lab block ho to poora block timetable, lunch aur teacher availability ke andar fit ho.

**Example:** agar Prof_Smith Monday Period 1 par unavailable hain, to unke session ka us waqt wala option domain mein nahi aana chahiye. Search ko phir sirf bache hue options try karne padte hain.

In [ ]:
# Ek real session aur uske filtered choices dekhein
sample = problem.variables[0]
sample_domain = problem.get_domain_for_variable(sample)
print("Variable ID:", sample.var_id)
print("Class:", sample.class_name)
print("Subject:", sample.subject_code)
print("Teacher:", sample.teacher_name)
print("Lab session:", sample.is_lab)
print("Filtered domain size:", len(sample_domain))
print("Kuch allowed (day, period, room) choices:")
for slot in sample_domain[:10]:
    print(f"  ({slot.day}, P{slot.period}, {slot.room_name})")

## 3. Search ko sahi ya galat kaise pata chalta hai?

`ConstraintChecker` har candidate timetable ko evaluate karta hai. **Hard constraints** tootne par timetable invalid hai; **soft constraints** quality score ko affect karte hain.

### Hard constraints

- Teacher, class ya room ka same waqt par double-book hona mana hai.
- Room ki capacity class strength ke liye sufficient ho.
- Lab subject lab room mein; theory classroom mein.
- Teacher unavailable slot mein class nahi.
- Teacher ka maximum weekly load cross nahi hota.
- Class ke lunch period mein lecture nahi.
- Linked multi-hour lab sessions same din, same room aur consecutive periods mein hon.

### Soft constraints

- Class ke din ke beech free gaps kam rakhna.
- Ek subject ke lectures alag-alag dinon mein spread karna.
- Lab periods ko consecutive rakhna.
- Teacher ke preferred slots ko respect karna.
- Teacher ko do se zyada back-to-back periods dene se bachana.

**Cost formula:** `1000 × hard violations + soft penalties`. Isliye hard conflict hataana soft score thoda improve karne se pehle aata hai. Example ke liye Small dataset mein ek teacher aur room ko ek waqt mein do alag classes dene par teacher-clash aur room-clash check trigger hoga.

In [ ]:
# Checker ka cost formula ek khaali/partial assignment par dekhein
empty_assignment = {}
initial_cost, initial_hard, initial_soft = checker.calculate_total_cost(empty_assignment)
print("Empty assignment cost:", initial_cost)
print("Hard violations:", initial_hard)
print("Soft penalty:", initial_soft)
print("\nNote: partial assignment par yeh sirf abhi assign hue sessions ko evaluate karta hai.")

## 4. Project ke chhe search algorithms

**A. Plain Backtracking (Baseline)**  
Variables fixed order mein aate hain. Ek valid slot assign hota hai; aage conflict/dead end aaya to pichhle session par lautkar doosra slot try hota hai. Jaise Monday P1 par teacher busy milne par next allowed slot try karna.

**B. Heuristic Backtracking (MRV + Degree + Forward Checking + LCV)**  
- **MRV:** sabse kam available choices wale session ko pehle schedule karo.
- **Degree tie-breaker:** equal choices hon to zyada doosre sessions ke saath conflict kar sakne wala session pehle.
- **Forward checking:** ek slot choose karne ke baad baaki sessions ke ab impossible options hatao; kisi ka domain khaali ho to turant backtrack.
- **LCV:** pehle woh slot try karo jo doosre sessions ke kam options block karta hai.

**C. Min-Conflicts**  
Ek poora initial candidate banata hai, phir conflict wale session ko aise slot par le jaata hai jahan clashes sabse kam hon. Stuck/plateau par random walk ya restart nayi direction try karte hain. Consecutive labs ko pair ke roop mein move karne ki koshish hoti hai.

**D. Simulated Annealing**  
Current timetable se neighbour banata hai (single session move, do sessions swap, ya lab pair move). Better cost wala move accept hota hai; kabhi-kabhi worse move bhi accept hota hai. Temperature cooling ke saath worse move accept hone ka chance kam hota hai.

**E. Hybrid CSP + Simulated Annealing**  
Pehle heuristic CSP se feasible, zero-hard-conflict timetable dhoondhta hai. Phir soft penalties improve karne ke liye annealing karta hai. CSP fail/timeout ho to code SA fallback use kar sakta hai.

**F. Hill Climbing**  
Candidate neighbours mein se best cost improvement leta hai. Agar sampled neighbours mein koi behtar na ho to local optimum par rukta hai; isi wajah se kabhi hard conflicts bache reh sakte hain.

### Visualisation ko kaise padhein

Local search ke plots mein **X-axis iteration/step** aur **Y-axis cost** hai; cost girna improvement dikhata hai. Backtracking solvers cost history record nahi karte, isliye unke panel mein **expanded search nodes** aur feasibility/status dikhaya jaayega. Comparison plots mein runtime, hard violations aur soft penalty alag compare honge. Ek run universal winner prove nahi karta—seed, parameters aur dataset se result badal sakta hai.

In [ ]:
# Chhe algorithms run karein. Bada dataset ho to is cell ko chalne mein zyada waqt lagega.
# Chhoti limits demo ko manageable rakhti hain; zarurat par badha sakte hain.

solvers = [
    ("Backtracking (Baseline)", PlainBacktrackingSolver(problem, timeout_seconds=8)),
    ("Backtracking (+ Heuristics)", HeuristicBacktrackingSolver(problem, timeout_seconds=8)),
    ("Min-Conflicts", MinConflictsSolver(
        problem, max_iterations=600, max_restarts=2,
        restart_patience=250, random_seed=42,
    )),
    ("Simulated Annealing", SimulatedAnnealingSolver(
        problem, max_total_steps=600, random_seed=42,
    )),
    ("Hybrid CSP + SA", HybridCSPSolver(
        problem, sa_max_steps=300, random_seed=42,
    )),
    ("Hill Climbing", HillClimbingSolver(
        problem, max_iterations=250, sample_neighborhood_size=20,
        random_seed=42,
    )),
]

results = []
for label, solver in solvers:
    started = time.perf_counter()
    result = solver.solve()
    elapsed = time.perf_counter() - started
    results.append((label, result))
    print(
        f"{label}: success={result.success}, hard={result.hard_violations}, "
        f"soft={result.soft_penalty}, nodes/steps={result.nodes_expanded}, "
        f"elapsed={elapsed:.2f}s"
    )

## 5. Dataset visualisations

Pehla chart batata hai ki har class ke kitne sessions schedule karne hain. Doosra chart session variables ke filtered domain sizes ka distribution dikhata hai. Domain chhota ho to session ke options kam hain—heuristic CSP ka MRV isi tarah ke constrained variables pehle pick karta hai.

In [ ]:
class_names = list(problem.classes)
session_counts = [
    sum(v.class_name == class_name for v in problem.variables)
    for class_name in class_names
]
domain_sizes = [len(problem.get_domain_for_variable(v)) for v in problem.variables]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(class_names, session_counts, color="#3B82F6")
axes[0].set_title("Har class ke kitne sessions schedule karne hain")
axes[0].set_ylabel("Session variables")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y", alpha=0.25)

axes[1].hist(domain_sizes, bins=16, color="#10B981", edgecolor="white")
axes[1].set_title("Har session ke valid options (domain size)")
axes[1].set_xlabel("Filtered day-period-room choices")
axes[1].set_ylabel("Sessions ki ginti")
axes[1].grid(axis="y", alpha=0.25)

fig.suptitle(f"Dataset overview — {problem.name}")
fig.tight_layout()
plt.show()

## 6. Har search ka apna result visualisation

Is figure mein chhe panels hain. Min-Conflicts, Simulated Annealing, Hybrid aur Hill Climbing ke panels mein solver ki recorded cost history plot hogi. Dono Backtracking panels mein expanded nodes dikhte hain, kyunki un solvers ka code per-step cost history store nahi karta. Panel par hard count, soft penalty, runtime aur feasible/incomplete status bhi annotate hote hain.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for ax, (label, result) in zip(axes, results):
    history = getattr(result, "cost_history", None)
    if history:
        ax.plot(range(len(history)), history, color="#2563EB", linewidth=1.8)
        ax.set_xlabel("Iteration / step")
        ax.set_ylabel("Cost (1000 x hard + soft)")
        ax.set_title(f"{label}: cost ka badlav")
        ax.grid(alpha=0.25)
    else:
        ax.bar(["Expanded nodes"], [result.nodes_expanded], color="#F59E0B")
        ax.set_ylabel("Search nodes")
        ax.set_title(f"{label}: search effort")
        ax.grid(axis="y", alpha=0.25)

    status = "Feasible" if result.success else "Incomplete / conflicts"
    ax.text(
        0.02, 0.96,
        f"{status}\nHard: {result.hard_violations} | Soft: {result.soft_penalty}\n"
        f"Time: {result.time_taken:.2f}s",
        transform=ax.transAxes, va="top", ha="left", fontsize=9,
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
    )

fig.suptitle("Har search ka result: convergence ya search effort", fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## 7. Chhe algorithms ka comparison

Runtime, hard violations aur soft penalty ko alag charts mein compare karo. **Hard = 0** aur `success=True` feasible result ka signal hai. Feasible schedules ke beech soft penalty jitni kam, utni quality is project ke soft rules ke mutabik behtar. Runtime aur node/step count search effort samajhne mein madad karte hain.

In [ ]:
labels = [label for label, _ in results]
run_results = [result for _, result in results]
x = list(range(len(labels)))

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
axes[0].bar(x, [r.time_taken for r in run_results], color="#8B5CF6")
axes[0].set_ylabel("Seconds")
axes[0].set_title("Execution time")

axes[1].bar(x, [r.hard_violations for r in run_results], color="#EF4444")
axes[1].set_ylabel("Violation count")
axes[1].set_title("Hard violations (0 = no counted hard conflicts)")

axes[2].bar(x, [r.soft_penalty for r in run_results], color="#14B8A6")
axes[2].set_ylabel("Penalty points")
axes[2].set_title("Soft penalty (lower is better among feasible schedules)")
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=20, ha="right")

for ax in axes:
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Search algorithms ka side-by-side comparison", fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## 8. Presentation ke liye short conclusion

Aap class mein is tarah summarize kar sakte ho:

> “Humne timetable data ko CSP ke variables, domains aur constraints mein model kiya. Dataset mein classes, teachers, subjects, rooms aur time slots hain. Hard constraints valid timetable banate hain; soft constraints timetable ko convenient banate hain. Backtracking systematic search karta hai, heuristics CSP search ko guide/prune karti hain, local search candidate schedule ko improve karta hai, aur Hybrid pehle feasibility phir quality optimize karta hai. Result ko hard violations, soft penalty, runtime aur search effort se compare karte hain.”

**Result samajhne ki savdhani:** `success=False` ya partial result aaye to soft penalty akela quality comparison ke liye kaafi nahi. Solver ka time limit, random seed aur iteration parameters result ko affect karte hain. Isliye presentation mein chart ke saath dataset aur parameters bhi mention karein.